# 2512 resistor on FR4

Created with Grok Build

This page is a transient heat-conduction model of one 2512 chip resistor on an FR4 coupon. The resistor solid is the KiCad library part `R_2512_6332Metric.step`. The dissipation is 0.1 W, spread uniformly through that solid. Two 1 oz copper traces, each 20 mil wide and 1 inch long, leave the end caps. Salome 9.16 meshes the assembly. ElmerSolver 26.2 integrates the heat equation for 585 s. ParaView 5.10 draws the temperature, including a movie of the whole run.


## The part

The solid is one body from the KiCad 3D library, file `R_2512_6332Metric.step`, header date 2018-01-04. KiCad StepUp wrote it in FreeCAD. The library is CC-BY-SA 4.0 with the KiCad library exception. Salome reads the STEP in metres. The mesh script scales it by 1000 and works in millimetres.

The body is 6.30 mm long, 3.20 mm wide, and 0.60 mm high. Its volume is 11.0970 mm³. The flat metal on the underside of each end cap spans $x$ from −3.0901 mm to −2.6099 mm on the left and from 2.6099 mm to 3.0901 mm on the right, with $y$ from −1.6001 mm to 1.6001 mm. Each of those faces has area 1.5360 mm². The ceramic between the caps sits about 0.06 mm higher, so the middle of the body does not touch the board. The outer 0.06 mm of the silhouette is a fillet.

That outline matches the Vishay D/CRCW e3 CRCW2512 body in document 20035, revision 14-Apr-2026: length 6.3 ± 0.20 mm, width 3.15 ± 0.15 mm, height 0.6 ± 0.10 mm. The STEP width of 3.20 mm is inside the width tolerance. Vishay lists a 1.0 W rating at 70 °C and a film temperature of 155 °C. The construction is a cermet film and glass on high-grade alumina, with a nickel barrier and matte tin.

The STEP contains the outline only. The film, the glass, the nickel, and the tin are absent, so the 0.1 W fills the one solid the file does contain. That solid is given the conductivity of CoorsTek AD-96 96% alumina, 24.7 W/(m·K), from ASTM C408. CoorsTek also lists this grade as ADD-96. The density is 3720 kg/m³, from ASTM C20, and the specific heat is 880 J/(kg·K) at 100 °C, from ASTM E1269. Vishay calls the substrate high-grade alumina and does not name this grade. The tin terminations use the same values because they are part of the same solid.


## Traces and coupon

Each trace is 20 mil wide, which is 0.508 mm, and 0.035 mm thick, which is 1 oz copper. The 1 inch length, 25.4 mm, is measured from the outer edge of the flat end-cap bottom. Under the cap the trace covers that bottom face in $x$ and stays 20 mil wide in $y$. The Vishay reflow land is a larger pad, with $G = 4.75$ mm, $Y = 1.45$ mm, $X = 3.50$ mm, and $Z = 7.65$ mm. This model keeps the 20 mil width for the whole run.

The left trace runs from $x = -28.4901$ mm to $x = -2.6099$ mm. The right trace runs from $x = 2.6099$ mm to $x = 28.4901$ mm. Each trace has volume 0.4603 mm³. The chip is translated so its lowest face lies on top of the foil, at $z = 0.035$ mm. The shared face between the chip and the copper is 0.2438 mm² at each end. The shared face between the FR4 and the copper is 13.1471 mm² on each trace. The 0.035 mm gap under the end cap, beside the 20 mil trace, is left out of the mesh.

The coupon size is a choice for this case. Document 20035 does not give the dimensions of the board used for the 1 W rating. The coupon here is 1.6 mm FR4, from $z = -1.6$ mm to $z = 0$. It extends 2 mm past each trace end and 5 mm past each side of the chip, so the rectangle runs from $x = -30.4901$ mm to $x = 30.4901$ mm and from $y = -6.6001$ mm to $y = 6.6001$ mm, about 60.98 mm by 13.20 mm. The FR4 volume is 1287.9213 mm³.

Copper conductivity is 401 W/(m·K). FR4 conductivity is 0.4 W/(m·K) in every direction, the Isola 370HR through-thickness value from ASTM E1952. The glass fabric makes a real board anisotropic. This case uses that one published number in every direction.

Open surfaces lose heat with a transfer coefficient of 10 W/(m²·K) to air at 25 °C. That coefficient is a mid-range value for natural convection, chosen for this model.


## Power

The 0.1 W is uniform in the chip volume, $1.10970223\times 10^{-8}\,\mathrm{m}^3$. Elmer multiplies `Heat Source` by `Density`. With the alumina density of 3720 kg/m³, the source in the case file is the power divided by the mass of the chip:

$$
h = \frac{0.1}{3720 \times 1.10970223\times 10^{-8}} = 2.42242646\times 10^{3}\,\mathrm{W/kg}.
$$

The same power divided by the volume is $9.01142643\times 10^{6}\,\mathrm{W/m}^3$. The product of density, the source, and the volume is 0.100000 W. The cell repeats the division. `case.sif` uses the source in W/kg.

An earlier steady run of this same mesh put 0.5 W in the chip and peaked at 182.953 °C. That rise is 157.953 K. One fifth of it is 31.591 K, which would put a 0.1 W peak near 56.59 °C. The transient on this page reaches 56.53 °C at 585 s. The 0.5 W peak sits above the 155 °C film temperature, so the dissipation on this page is 0.1 W.


In [1]:
volume_m3 = 1.10970223e-08
power_w = 0.1
density = 3720.0
source_w_m3 = power_w / volume_m3
source_w_kg = power_w / (density * volume_m3)
print(f"volume {volume_m3:.8e} m^3")
print(f"heat source {source_w_m3:.8e} W/m^3")
print(f"heat source {source_w_kg:.8e} W/kg")
print(f"power {density * source_w_kg * volume_m3:.6f} W")


volume 1.10970223e-08 m^3
heat source 9.01142643e+06 W/m^3
heat source 2.42242646e+03 W/kg
power 0.100000 W


## Mesh

`mesh_r2512.py` imports the STEP, builds the two traces and the coupon, and partitions the assembly into four solids: the chip, the left trace, the right trace, and the FR4. The partition uses rectangle tools on the planes $z = 0$ and $z = 0.035$ mm, with a fuzzy tolerance of $1.0\times 10^{-4}$ mm, so the foil stays 0.035 mm thick.

Netgen 1D-2D-3D with a maximum element of 0.35 mm and a minimum of 0.08 mm produced a mesh of linear tetrahedra. The mesh has 56040 nodes, 63619 faces, and 244247 tetrahedra. The chip has 16146 tetrahedra, the left trace 8932, the right trace 9440, and the FR4 209729. The air skin is a face group of 57622 triangles. Salome writes `r2512.unv` with coordinates in millimetres.

ElmerGrid converts that file and multiplies every coordinate by 0.001, so the mesh Elmer solves is in metres. The option `-autoclean` keeps the four bodies. It also writes an internal face group, `bnry6`, on faces that have two parent elements. Those faces lie inside the assembly, on shared nodes. The air boundary condition targets the air skin only. Heat conducts across `bnry6`. The solved field shows that connection: the chip is hot, and the traces and the board sit above the air temperature.


## Heat equation

The solver is `HeatSolve` / `HeatSolver`, transient, BDF of order 2, with one nonlinear iteration. The schedule is 10 steps of 0.5 s, then 20 steps of 5 s, then 24 steps of 20 s. Elmer writes a result on every step, 54 frames in all, and the last time is 585 s. Every body starts at 25 °C. The linear solver is BiCGStab with ILU1, a tolerance of $1.0\times 10^{-10}$, and a limit of 2000 iterations. The wall time was 149.15 s. The series is `mesh/case_t0001.vtu` through `mesh/case_t0054.vtu`. The last frame is kept beside this notebook as `case_final.vtu`.

HeatSolve multiplies `Heat Source` by `Density`, and `Heat Capacity` is the specific heat. The chip uses the AD-96 pair above, 3720 kg/m³ and 880 J/(kg·K). Copper is 8960 kg/m³ and 385 J/(kg·K), the room-temperature values for pure copper. Isola does not publish a density or a specific heat for 370HR. This case uses 1850 kg/m³, the usual FR-4 specific gravity of 1.85, and 1000 J/(kg·K). Those two numbers set how fast the board warms. The conductivity stays the Isola through-thickness value, 0.4 W/(m·K), in every direction.

The body and boundary names are the `$` names in `mesh/mesh.names`: `chip`, `copper_left`, `copper_right`, `fr4`, and `air`. Elmer loads that file after it parses `case.sif`, so the case includes `mesh/mesh.names` before the body blocks.

The case file is the record of the materials, the source, the time steps, and the air boundary.

```{literalinclude} case.sif
:language: text
```


## Temperature

At 0.5 s the field runs from 25.00 °C to 26.30 °C. The hottest node is 34.97 °C at 5 s, 48.45 °C at 30 s, 51.31 °C at 60 s, and 53.24 °C at 105 s. At 585 s the field runs from 25.72 °C to 56.53 °C. The color scale on every frame, and on both stills, is locked from 25.00 °C to 56.53 °C. No earlier frame is hotter than the last.

Over the last 20 s the hottest node rose from 56.524 °C to 56.534 °C. The solver norm changed by 0.009 on that step, a relative change of $2.7\times 10^{-4}$. The chip has leveled, and the coupon is still creeping. A rise of 31.53 K at 0.1 W is about 315 K/W for this mounting. The temperature drop sits at the two contacts, 0.2438 mm² each, where the heat enters the 20 mil traces, and then in the FR4. The 56.53 °C peak is the one-fifth scaling of the earlier steady peak, 182.953 °C, on this same mesh.

56.53 °C is the hottest node in this mesh at 585 s. Vishay's film limit is 155 °C on the 1 W test. This page uses 0.1 W, 20 mil traces, a coupon of about 61 mm by 13 mm, and a convection coefficient of 10 W/(m²·K). The film is absent from the STEP. The temperature belongs to this model.

The clip is 54 frames at 5 frames per second, so it runs 10.8 s. The first 10 frames are 0.5 s apart, the next 20 are 5 s apart, and the last 24 are 20 s apart. The clock is the simulation time.

```{figure} images/temperature_evolution.mp4
:alt: Movie of the coupon temperature from 0.5 s to 585 s
:width: 900
:align: center
:enumerated: false
:class: clickable-figure

Temperature from 0.5 s to 585 s. The color bar stays at 25.00 °C to 56.53 °C. The chip warms first, and the heat then follows the two traces into the coupon.
```

```{figure} images/temperature.png
:alt: Temperature on the 2512 coupon at 585 s, with the color bar on the right
:width: 900
:align: center
:enumerated: false
:class: clickable-figure

Temperature on the coupon at 585 s. The color bar runs from 25.00 °C to 56.53 °C and sits to the right of the board. The chip is the hot body in the center. Heat follows the two traces, and the rest of the coupon stays near the air temperature.
```

```{figure} images/temperature_chip.png
:alt: Close view of the 2512 chip at 585 s, both end caps, and the 20 mil traces
:width: 900
:align: center
:enumerated: false
:class: clickable-figure

The chip and the inner ends of the traces at 585 s. The end-cap steps are the KiCad solid. Each trace stays 20 mil wide where it meets the part, and the heat spreads from those traces into the FR4.
```


## Files

`mesh_r2512.py` builds the geometry, writes `case.sif`, and writes the UNV. The mesh was kept for the 0.1 W run. `render_movie.py` draws one picture per saved step and writes the two stills from the last step. `ffmpeg` joins those pictures into `images/temperature_evolution.mp4`. `render_temperature.py` can redraw the two stills from `case_final.vtu`. `elmer.log` is the solver log. `mesh_r2512.log` is the Salome log, including the volumes and the contact areas. The heat-source line in that log is the 0.5 W value written when the mesh was built. The case file beside this notebook is the later 0.1 W transient. The mesh script stays beside this notebook. `r2512.unv` is the Salome mesh.

From this directory:

```text
salome --keep-paths shell -- python3 mesh_r2512.py
ElmerGrid 8 2 r2512.unv -scale 1.0e-3 1.0e-3 1.0e-3 -autoclean -out mesh
ElmerSolver case.sif
```

On this machine Salome is the native 9.16.0 Debian 13 build. Both unpacked library directories go on `LD_LIBRARY_PATH` before that launch, and the `--` is part of the command. ElmerGrid and ElmerSolver 26.2 ran in the kicad-tools container. Each VTU lands in `mesh/` even though `Post File = case.vtu`. ParaView 5.10 draws the frames from that series, and this command turns the frames into the movie:

```text
ffmpeg -y -framerate 5 -i frame_%04d.png -c:v libx264 -pix_fmt yuv420p -crf 18 images/temperature_evolution.mp4
```
